## Celda 0 — Verificación de ZIPs en Drive

> **Versión: v3.1**

Confirma que los **13 ZIPs de Drive** y los **2 links de TeraBox** están listos antes de entrenar.


In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive', force_remount=False)

# ─── Configuración de rutas ───────────────────────────────────
DRIVE_ZIPS_DIR = '/content/drive/MyDrive/ExpoEscom/dataset_zips'

DRIVE_CLASSES = [
    'aot', 'barbie', 'ben_10', 'bleach', 'doraemon',
    'dragon_ball', 'naruto', 'one_piece',
    'pokemon', 'simpson', 'star_wars', 'unshowmas', 'yugioh'
]

TERABOX_LINKS = {
    'hora_de_aventura': 'https://1024terabox.com/s/1VU4O54W0U4P-9BY2LoikGw',
    'otra':             'https://1024terabox.com/s/1AbrV_1YFKGgauktw2oeQgw',
}

# ─── Verificación ────────────────────────────────────────────
print("=" * 60)
print("📂 VERIFICACIÓN — 13 Drive + 2 TeraBox")
print("=" * 60)

errors = []

print(f"\n📁 Drive ({len(DRIVE_CLASSES)} ZIPs esperados):")
if not os.path.isdir(DRIVE_ZIPS_DIR):
    errors.append(f"Carpeta Drive no existe: {DRIVE_ZIPS_DIR}")
    print(f"  ❌ Carpeta no encontrada: {DRIVE_ZIPS_DIR}")
    print("     Crea la carpeta en Drive y sube los 13 ZIPs.")
else:
    for clase in DRIVE_CLASSES:
        zip_path = os.path.join(DRIVE_ZIPS_DIR, clase + '.zip')
        if os.path.isfile(zip_path):
            size_mb = os.path.getsize(zip_path) / 1e6
            print(f"  ✅ {clase:<22s} ({size_mb:.0f} MB)")
        else:
            errors.append(f"Falta Drive ZIP: {clase}.zip")
            print(f"  ❌ {clase:<22s} NO ENCONTRADO")

print(f"\n☁️  TeraBox ({len(TERABOX_LINKS)} links directos):")
for clase, url in TERABOX_LINKS.items():
    print(f"  ✅ {clase:<22s} {url}")

print()
if errors:
    for e in errors:
        print(f"  ⚠️  {e}")
    raise ValueError(
        f"\n{len(errors)} problema(s) pendiente(s). "
        "Corrígelos y vuelve a correr esta celda."
    )

print("✅ Todo listo — continúa con la Celda 1.")

# ExpoEscom — Entrenamiento v3.1 (Colab Pro+ · rápido)

**Objetivo: bajar de ~13-16h a ~1-2h** sin tirar datos.

**Por qué era lento antes:** pasar 1.6M imágenes por el backbone completo en cada época (×18). Con el backbone congelado eso es trabajo repetido.

**Las 3 palancas grandes de esta versión:**
1. **Feature caching (Fase 1):** el backbone congelado da *siempre* el mismo vector por imagen. Se calcula **una sola pasada**, se guarda en disco (~4 GB) y la cabeza se entrena sobre esos vectores → de horas a minutos, usando TODAS las imágenes.
2. **Fine-tuning sobre subconjunto (Fase 2):** afinar 3 capas no necesita 100k/clase; con ~20k/clase balanceadas se obtiene casi la misma ganancia.
3. **Validación capada** (~1.5k/clase) — suficiente para un F1 estable, en vez de validar 330k imágenes cada época.

**Infra Pro+:** AMP (mixed precision), `channels_last`, TF32, batch/workers por GPU, `cudnn.benchmark`.

> ⚠️ **El cuello de botella real es mover 1.6M archivos desde Drive.** `copytree` archivo-por-archivo puede tardar HORAS. Sube tu dataset como **un solo `dataset.zip`** a Drive una vez; la celda 4 lo extrae local en minutos.

---
**Changelog**
- **v3.1** — descarga de 15 ZIPs (uno por clase) al storage local de Colab (15 lecturas, no millones de archivos); 'otra' se expande en 3 subclases (17 clases); nombres de clase corregidos (aot, unshowmas, yugioh)
- **v3.0** — feature caching para Fase 1, fine-tuning sobre subconjunto, val capada, channels_last/TF32, extracción desde .zip
- **v2.1** — soporte Pro+: AMP, batch/workers adaptativos, prefetch
- **v2.0** — CrossEntropyLoss, split estratificado, sampler ponderado, F1 por clase

## Celda 1 — Setup

In [ ]:
!pip install scikit-learn -q

import os, time, warnings, math
from collections import defaultdict
import numpy as np
import torch
import torch.nn as nn
import torchvision.models as models
import torchvision.transforms as transforms
from torch.utils.data import Dataset, DataLoader, TensorDataset
from PIL import Image
from sklearn.metrics import f1_score
import matplotlib.pyplot as plt
from google.colab import drive

warnings.filterwarnings('ignore')
drive.mount('/content/drive', force_remount=False)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"PyTorch  : {torch.__version__}")
print(f"Device   : {DEVICE}")

if not torch.cuda.is_available():
    print("\n⚠️  NO TIENES GPU ACTIVA")
    print("   Entorno de ejecución → Cambiar tipo → GPU → A100/L4/V100 → Guardar")
else:
    print(f"GPU      : {torch.cuda.get_device_name(0)}")
    print(f"VRAM     : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## Celda 2 — Configuración

In [ ]:
# ════════════════════════════════════════════════════════════
# ExpoEscom — Entrenamiento v3.1 | Colab Pro+ | Feature Caching
# ════════════════════════════════════════════════════════════
VERSION = '3.1'
print(f"\n{'═'*60}")
print(f"📦 ExpoEscom Entrenamiento v{VERSION}")
print(f"{'═'*60}\n")

# ════ RUTAS ══════════════════════════════════════════════════
DRIVE_ZIPS_DIR = '/content/drive/MyDrive/ExpoEscom/dataset_zips'
DATASET_ROOT   = '/content/dataset_local'
SAVE_DIR       = '/content/drive/MyDrive/ExpoEscom/models'
MODEL_NAME     = 'cartoon_v3.pt'
FEATURE_CACHE  = '/content/features_cache.pt'

# ════ TERABOX — 2 clases (links compartidos, sin cookies) ════
TERABOX_LINKS = {
    'hora_de_aventura': 'https://1024terabox.com/s/1VU4O54W0U4P-9BY2LoikGw',
    'otra':             'https://1024terabox.com/s/1AbrV_1YFKGgauktw2oeQgw',
}

# 13 clases de Drive + 2 de TeraBox
DRIVE_CLASSES = [
    'aot', 'barbie', 'ben_10', 'bleach', 'doraemon',
    'dragon_ball', 'naruto', 'one_piece',
    'pokemon', 'simpson', 'star_wars', 'unshowmas', 'yugioh'
]
TOP_FOLDERS = DRIVE_CLASSES + list(TERABOX_LINKS.keys())

# ════ LÍMITES DE DATOS ═══════════════════════════════════════
MAX_PER_CLASS = 100_000
MAX_OTRA      = 250_000
VAL_PER_CLASS = 1_500

# ════ FASE 1 ═════════════════════════════════════════════════
NUM_EPOCHS_P1 = 20
LR_PHASE1     = 1e-3
HEAD_BATCH    = 4096

# ════ FASE 2 ═════════════════════════════════════════════════
FT_PER_CLASS  = 20_000
NUM_EPOCHS_P2 = 4
LR_PHASE2     = 1e-5
UNFREEZE_N    = 3

# ════ AJUSTE AUTOMÁTICO SEGÚN GPU ════════════════════════════
gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'
if   'A100' in gpu_name: BATCH_SIZE, NUM_WORKERS = 256, 8
elif 'V100' in gpu_name: BATCH_SIZE, NUM_WORKERS = 160, 8
elif 'L4'   in gpu_name: BATCH_SIZE, NUM_WORKERS = 128, 8
else:                    BATCH_SIZE, NUM_WORKERS = 64, 4

USE_AMP = torch.cuda.is_available()
torch.backends.cudnn.benchmark = True
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

# ════ RUIDO ══════════════════════════════════════════════════
NOISE_PARENTS = ('otra', 'noise', 'ruido')
def is_noise(class_name):
    return class_name.lower().split('/')[0] in NOISE_PARENTS

os.makedirs(SAVE_DIR, exist_ok=True)
SAVE_PATH = os.path.join(SAVE_DIR, MODEL_NAME)
print(f"✅ Config | GPU: {gpu_name}")
print(f"   Batch={BATCH_SIZE} | Workers={NUM_WORKERS} | AMP={USE_AMP}")
print(f"   Drive ZIPs    : {DRIVE_ZIPS_DIR}")
print(f"   TeraBox clases: {list(TERABOX_LINKS.keys())}")
print(f"   Dataset local : {DATASET_ROOT}")
print(f"   Guardado en   : {SAVE_PATH}")

## Celda 3 — Descargar los 15 ZIPs a local y extraer

Copia los 15 ZIPs de Drive al storage local de Colab (**15 lecturas
secuenciales**, no millones de archivos) y los extrae. A partir de aquí,
**el entrenamiento solo usa el disco local de Colab; Drive no se vuelve a tocar.**

In [ ]:
import shutil, zipfile

os.makedirs(DATASET_ROOT, exist_ok=True)
LOCAL_ZIPS = '/content/_zips_tmp'
os.makedirs(LOCAL_ZIPS, exist_ok=True)

print("═" * 60)
print("⬇️  DESCARGA + EXTRACCIÓN (13 Drive + 2 TeraBox → local)")
print("═" * 60)

t_all = time.time()

# ── 13 de Drive ──────────────────────────────────────────────
print("\n📁 Desde Drive:")
for clase in DRIVE_CLASSES:
    dst_dir = os.path.join(DATASET_ROOT, clase)
    if os.path.isdir(dst_dir) and any(os.scandir(dst_dir)):
        print(f"  ✅ {clase:<22s} ya extraído (se salta)")
        continue

    src_zip = os.path.join(DRIVE_ZIPS_DIR, clase + '.zip')
    if not os.path.isfile(src_zip):
        raise FileNotFoundError(f"Falta {src_zip} — corre la Celda 0 primero.")

    size_mb = os.path.getsize(src_zip) / 1e6
    t0 = time.time()
    local_zip = os.path.join(LOCAL_ZIPS, clase + '.zip')
    shutil.copy(src_zip, local_zip)
    cp = time.time() - t0

    t0 = time.time()
    with zipfile.ZipFile(local_zip) as zf:
        zf.extractall(DATASET_ROOT)
    ex = time.time() - t0

    os.remove(local_zip)
    print(f"  📦 {clase:<22s} {size_mb:6.0f} MB | copia {cp:4.0f}s | extrae {ex:4.0f}s")

# ── 2 de TeraBox (links compartidos, sin cookies) ────────────
print("\n☁️  Desde TeraBox:")
for clase, url in TERABOX_LINKS.items():
    dst_dir = os.path.join(DATASET_ROOT, clase)
    if os.path.isdir(dst_dir) and any(os.scandir(dst_dir)):
        print(f"  ✅ {clase:<22s} ya extraído (se salta)")
        continue

    print(f"  ⬇️  {clase:<22s} descargando...")
    local_zip = os.path.join(LOCAL_ZIPS, clase + '.zip')

    t0 = time.time()
    ret = os.system(f'wget -q "{url}" -O "{local_zip}"')
    dl = time.time() - t0

    if ret != 0 or not os.path.isfile(local_zip):
        raise RuntimeError(
            f"Error descargando {clase} desde TeraBox.\n"
            f"Verifica que el link siga activo: {url}"
        )

    t0 = time.time()
    with zipfile.ZipFile(local_zip) as zf:
        zf.extractall(DATASET_ROOT)
    ex = time.time() - t0

    size_mb = os.path.getsize(local_zip) / 1e6
    os.remove(local_zip)
    print(f"  📦 {clase:<22s} {size_mb:6.0f} MB | descarga {dl:4.0f}s | extrae {ex:4.0f}s")

shutil.rmtree(LOCAL_ZIPS, ignore_errors=True)
print(f"\n✅ Todo en local en {time.time()-t_all:.0f}s → {DATASET_ROOT}")

cand = os.path.join(DATASET_ROOT, 'dataset')
if os.path.isdir(cand):
    DATASET_ROOT = cand
print(f"DATASET_ROOT = {DATASET_ROOT}")

## Celda 4 — Diagnóstico del dataset (local)

Cuenta imágenes por clase ya en local. **'otra' se expande** en sus
subcarpetas (cartoons_anime / noise / real_life) → estructura de **17 clases**,
igual que el modelo local.

In [ ]:
def detect_classes(root):
    """Subcarpetas con imágenes = clases. 'otra' se expande en sus
    subcarpetas; las clases de ruido van al final."""
    cartoons, noise = [], []
    for name in sorted(os.listdir(root)):
        d = os.path.join(root, name)
        if not os.path.isdir(d):
            continue
        if is_noise(name):
            subs = [s for s in sorted(os.listdir(d))
                    if os.path.isdir(os.path.join(d, s))]
            noise += [f"{name}/{s}" for s in subs] if subs else [name]
        else:
            cartoons.append(name)
    return cartoons + noise


def count_imgs_in_dir(dir_path):
    n = 0
    for _, _, files in os.walk(dir_path):
        n += sum(1 for f in files if f.lower().endswith(('.jpg', '.jpeg', '.png', '.webp')))
    return n


print("═" * 60)
print("📁 DIAGNÓSTICO DEL DATASET (local)")
print("═" * 60)

CLASSES = []
for cls in detect_classes(DATASET_ROOT):
    cls_dir = os.path.join(DATASET_ROOT, *cls.split('/'))
    n       = count_imgs_in_dir(cls_dir)
    limite  = MAX_OTRA if is_noise(cls) else MAX_PER_CLASS
    if n == 0:
        print(f"  ⚠️  {cls:<22s} → carpeta vacía (se salta)")
        continue
    CLASSES.append(cls)
    print(f"  ✅ {cls:<22s} → {n:7,} imgs, usará {min(n, limite):,}")

NUM_CLASSES = len(CLASSES)
print(f"\n{'═'*60}")
print(f"Clases activas : {NUM_CLASSES}")
print(f"Lista          : {CLASSES}")

## Celda 5 — Dataset + recolección de muestras

In [ ]:
class CartoonDataset(Dataset):
    def __init__(self, samples, transform=None):
        self.samples   = samples
        self.transform = transform

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        try:
            img = Image.open(path).convert('RGB')
        except Exception:
            img = Image.new('RGB', (224, 224), (128, 128, 128))
        if self.transform:
            img = self.transform(img)
        return img, torch.tensor(label, dtype=torch.long)


def build_samples(root_dir, classes, max_per_class, max_otra):
    all_samples = []
    for cls_idx, cls_name in enumerate(classes):
        cls_dir = os.path.join(root_dir, *cls_name.split('/'))
        if not os.path.exists(cls_dir):
            continue
        limite = max_otra if is_noise(cls_name) else max_per_class
        imgs = []
        for root, _, files in os.walk(cls_dir):
            for f in files:
                if f.lower().endswith(('.jpg', '.jpeg', '.png', '.webp')):
                    imgs.append(os.path.join(root, f))
        np.random.shuffle(imgs)
        imgs = imgs[:limite]
        if not imgs:
            print(f"  ⚠️  {cls_name}: sin imágenes")
            continue
        for path in imgs:
            all_samples.append((path, cls_idx))
        print(f"  ✅ {cls_name:22s}: {len(imgs):,}")
    return all_samples


print("Construyendo lista de muestras...")
np.random.seed(42)
all_samples = build_samples(DATASET_ROOT, CLASSES, MAX_PER_CLASS, MAX_OTRA)
print(f"\nTotal: {len(all_samples):,} imágenes")

## Celda 6 — Split (validación capada por clase)

In [ ]:
by_class = defaultdict(list)
for s in all_samples:
    by_class[s[1]].append(s)

rng = np.random.default_rng(42)
train_samples, val_samples = [], []
for c, items in by_class.items():
    items = list(items)
    rng.shuffle(items)
    val_samples   += items[:VAL_PER_CLASS]
    train_samples += items[VAL_PER_CLASS:]

rng.shuffle(train_samples)
train_labels = np.array([s[1] for s in train_samples])
class_counts = np.bincount(train_labels, minlength=NUM_CLASSES).astype(float)

print(f"Train: {len(train_samples):,}  |  Val: {len(val_samples):,}")
print("\nImágenes de train por clase:")
for i, cls in enumerate(CLASSES):
    print(f"  {cls:22s}: {int(class_counts[i]):,}")

## Celda 7 — Transforms y DataLoaders

In [ ]:
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

# Aumentación SOLO para Fase 2 (fine-tuning con imágenes reales)
transform_train = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(0.5),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2, hue=0.1),
    transforms.RandomRotation(15),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

# Sin aumentación: para extraer features y para validar
transform_plain = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

loader_kwargs = dict(num_workers=NUM_WORKERS, pin_memory=True,
                     persistent_workers=(NUM_WORKERS > 0))
if NUM_WORKERS > 0:
    loader_kwargs['prefetch_factor'] = 4

# Loader para EXTRAER features de todo el train (sin aumentación, sin shuffle)
feat_loader = DataLoader(CartoonDataset(train_samples, transform_plain),
                         batch_size=BATCH_SIZE, shuffle=False, **loader_kwargs)

# Loader de validación con imágenes reales (Fase 2 y reporte final)
val_loader  = DataLoader(CartoonDataset(val_samples, transform_plain),
                         batch_size=BATCH_SIZE, shuffle=False, **loader_kwargs)

print(f"✅ Loaders listos | feat: {len(feat_loader)} batches | val: {len(val_loader)} batches")

## Celda 8 — Modelo

In [ ]:
class CartoonClassifier(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        backbone      = models.mobilenet_v2(weights=models.MobileNet_V2_Weights.DEFAULT)
        self.features = backbone.features
        self.avgpool  = nn.AdaptiveAvgPool2d((1, 1))
        self.head = nn.Sequential(
            nn.Linear(1280, 512),
            nn.ReLU(inplace=True),
            nn.Dropout(0.3),
            nn.Linear(512, num_classes),
        )
        for p in self.features.parameters():
            p.requires_grad = False

    def extract(self, x):
        x = self.features(x)
        x = self.avgpool(x)
        return torch.flatten(x, 1)      # [B, 1280]

    def forward(self, x):
        return self.head(self.extract(x))

    def unfreeze_last_n(self, n=3):
        for layer in list(self.features.children())[-n:]:
            for p in layer.parameters():
                p.requires_grad = True
        t = sum(p.numel() for p in self.parameters() if p.requires_grad)
        print(f"   🔥 {n} capas descongeladas → {t:,} params entrenables")


def save_best(state_dict, epoch, best_f1, phase, history):
    torch.save({
        'model_state_dict': state_dict,
        'classes'         : CLASSES,
        'num_classes'     : NUM_CLASSES,
        'best_f1'         : best_f1,
        'epoch'           : epoch,
        'phase'           : phase,
        'history'         : history,
    }, SAVE_PATH)


model = CartoonClassifier(NUM_CLASSES).to(DEVICE).to(memory_format=torch.channels_last)
total = sum(p.numel() for p in model.parameters())
print(f"✅ Modelo listo — {NUM_CLASSES} clases | {total:,} params")

## Celda 9 — Funciones (features, train, eval)

In [ ]:
criterion = nn.CrossEntropyLoss()
scaler    = torch.amp.GradScaler('cuda', enabled=USE_AMP)


def compute_metrics(y_true, y_pred):
    f1m = f1_score(y_true, y_pred, average='macro', zero_division=0)
    f1c = f1_score(y_true, y_pred, average=None,    zero_division=0)
    return f1m, f1c


@torch.no_grad()
def extract_features(model, loader, desc=''):
    # Una pasada por el backbone congelado -> vectores [N, 1280] en fp16
    model.eval()
    feats, labs, t0 = [], [], time.time()
    for i, (imgs, labels) in enumerate(loader):
        imgs = imgs.to(DEVICE, non_blocking=True).to(memory_format=torch.channels_last)
        with torch.autocast('cuda', dtype=torch.float16, enabled=USE_AMP):
            f = model.extract(imgs)
        feats.append(f.half().cpu())
        labs.append(labels)
        if (i + 1) % 50 == 0:
            done = (i + 1) * loader.batch_size
            print(f"  {desc}: {done:,} imgs | {done/(time.time()-t0):.0f} img/s", end='\r')
    print()
    return torch.cat(feats), torch.cat(labs)


def train_epoch(model, loader, optimizer):
    model.train()
    total = 0.0
    for i, (imgs, labels) in enumerate(loader):
        imgs   = imgs.to(DEVICE, non_blocking=True).to(memory_format=torch.channels_last)
        labels = labels.to(DEVICE, non_blocking=True)
        optimizer.zero_grad()
        with torch.autocast('cuda', dtype=torch.float16, enabled=USE_AMP):
            loss = criterion(model(imgs), labels)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        total += loss.item()
        if (i + 1) % 50 == 0:
            print(f"  batch {i+1}/{len(loader)}", end='\r')
    return total / len(loader)


@torch.no_grad()
def eval_epoch(model, loader):
    model.eval()
    total, gts, preds = 0.0, [], []
    for imgs, labels in loader:
        imgs   = imgs.to(DEVICE, non_blocking=True).to(memory_format=torch.channels_last)
        labels = labels.to(DEVICE, non_blocking=True)
        with torch.autocast('cuda', dtype=torch.float16, enabled=USE_AMP):
            logits = model(imgs)
            total += criterion(logits, labels).item()
        preds.append(logits.argmax(1).cpu().numpy())
        gts.append(labels.cpu().numpy())
    y_true, y_pred = np.concatenate(gts), np.concatenate(preds)
    f1m, f1c = compute_metrics(y_true, y_pred)
    return total / len(loader), f1m, f1c


print("✅ Funciones listas")

## Celda 10 — Extraer features (1 sola pasada) + cache

Lo más pesado de toda la corrida. Se hace **una vez** y se guarda en disco; si la sesión se cae, al re-correr se carga del cache.

In [ ]:
if os.path.exists(FEATURE_CACHE):
    print("✅ Cargando features del cache...")
    cache = torch.load(FEATURE_CACHE)
    train_feats, train_feat_labels = cache['train_feats'], cache['train_labels']
    val_feats,   val_feat_labels   = cache['val_feats'],   cache['val_labels']
else:
    print("Extrayendo features (backbone congelado)...")
    t0 = time.time()
    train_feats, train_feat_labels = extract_features(model, feat_loader, 'train')
    val_feats,   val_feat_labels   = extract_features(model, val_loader,  'val')
    torch.save({'train_feats': train_feats, 'train_labels': train_feat_labels,
                'val_feats': val_feats, 'val_labels': val_feat_labels}, FEATURE_CACHE)
    print(f"✅ Cache guardado en {FEATURE_CACHE} ({time.time()-t0:.0f}s)")

print(f"   Train feats: {tuple(train_feats.shape)} | Val feats: {tuple(val_feats.shape)}")

## Celda 11 — Fase 1: cabeza sobre features cacheadas (rápida)

In [ ]:
print("═" * 60)
print("🚀 FASE 1 — Cabeza sobre features cacheadas (sin tocar imágenes)")
print("═" * 60)

feat_train_loader = DataLoader(TensorDataset(train_feats, train_feat_labels),
                               batch_size=HEAD_BATCH, shuffle=True)
feat_val_loader   = DataLoader(TensorDataset(val_feats, val_feat_labels),
                               batch_size=HEAD_BATCH, shuffle=False)

# Loss ponderada por clase → compensa el desbalance de 'otra'
cls_w = class_counts.sum() / (NUM_CLASSES * np.clip(class_counts, 1, None))
criterion_w = nn.CrossEntropyLoss(
    weight=torch.tensor(cls_w, dtype=torch.float32, device=DEVICE))

optimizer_p1 = torch.optim.Adam(model.head.parameters(), lr=LR_PHASE1, weight_decay=1e-4)
scheduler_p1 = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer_p1, mode='max', factor=0.5, patience=2)

history    = {'train_loss': [], 'val_loss': [], 'val_f1': []}
best_f1, best_state = 0.0, None

for epoch in range(1, NUM_EPOCHS_P1 + 1):
    t0 = time.time()
    model.head.train()
    tl = 0.0
    for fb, lb in feat_train_loader:
        fb, lb = fb.float().to(DEVICE), lb.to(DEVICE)
        optimizer_p1.zero_grad()
        loss = criterion_w(model.head(fb), lb)
        loss.backward()
        optimizer_p1.step()
        tl += loss.item()
    tl /= len(feat_train_loader)

    model.head.eval()
    vl, gts, preds = 0.0, [], []
    with torch.no_grad():
        for fb, lb in feat_val_loader:
            out = model.head(fb.float().to(DEVICE))
            vl += criterion_w(out, lb.to(DEVICE)).item()
            preds.append(out.argmax(1).cpu().numpy())
            gts.append(lb.numpy())
    vl /= len(feat_val_loader)
    val_f1 = f1_score(np.concatenate(gts), np.concatenate(preds),
                      average='macro', zero_division=0)
    scheduler_p1.step(val_f1)

    history['train_loss'].append(tl); history['val_loss'].append(vl)
    history['val_f1'].append(val_f1)
    star = " ⭐" if val_f1 > best_f1 else ""
    print(f"[P1 {epoch:02d}/{NUM_EPOCHS_P1}] {time.time()-t0:.0f}s | "
          f"loss {tl:.4f}/{vl:.4f} | F1 {val_f1:.4f}{star}")

    if val_f1 > best_f1:
        best_f1    = val_f1
        best_state = {k: v.clone() for k, v in model.state_dict().items()}
        save_best(best_state, epoch, best_f1, 'P1', history)

print(f"\n🏁 Mejor F1 Fase 1: {best_f1:.4f}")

## Celda 12 — Fase 2: fine-tuning sobre subconjunto balanceado

In [ ]:
print("═" * 60)
print(f"🔥 FASE 2 — Fine-tuning {UNFREEZE_N} capas | {FT_PER_CLASS:,} imgs/clase")
print("═" * 60)

# Subconjunto balanceado de train (imágenes reales, con aumentación)
by_class_train = defaultdict(list)
for s in train_samples:
    by_class_train[s[1]].append(s)
rng2 = np.random.default_rng(0)
ft_samples = []
for c, items in by_class_train.items():
    items = list(items)
    rng2.shuffle(items)
    ft_samples += items[:FT_PER_CLASS]
rng2.shuffle(ft_samples)
print(f"Subconjunto fine-tuning: {len(ft_samples):,} imágenes")

ft_loader = DataLoader(CartoonDataset(ft_samples, transform_train),
                       batch_size=BATCH_SIZE, shuffle=True, **loader_kwargs)

model.load_state_dict(best_state)
model.unfreeze_last_n(UNFREEZE_N)
model = model.to(memory_format=torch.channels_last)

optimizer_p2 = torch.optim.Adam(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=LR_PHASE2, weight_decay=1e-5)
scheduler_p2 = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer_p2, T_max=NUM_EPOCHS_P2)

for epoch in range(1, NUM_EPOCHS_P2 + 1):
    t0 = time.time()
    train_loss = train_epoch(model, ft_loader, optimizer_p2)
    val_loss, val_f1, _ = eval_epoch(model, val_loader)
    scheduler_p2.step()

    history['train_loss'].append(train_loss); history['val_loss'].append(val_loss)
    history['val_f1'].append(val_f1)
    star = " ⭐" if val_f1 > best_f1 else ""
    print(f"[P2 {epoch:02d}/{NUM_EPOCHS_P2}] {time.time()-t0:.0f}s | "
          f"loss {train_loss:.4f}/{val_loss:.4f} | F1 {val_f1:.4f}{star}")

    if val_f1 > best_f1:
        best_f1    = val_f1
        best_state = {k: v.clone() for k, v in model.state_dict().items()}
        save_best(best_state, epoch, best_f1, 'P2', history)

print(f"\n🏆 Mejor F1 Final: {best_f1:.4f}")
print(f"   Guardado en   : {SAVE_PATH}")

## Celda 13 — Gráficas y F1 por clase

In [ ]:
model.load_state_dict(best_state)

# ─── Gráficas ─────────────────────────────────────────────────
total_ep = len(history['val_f1'])
ep_r     = range(1, total_ep + 1)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle(f'ExpoEscom v3 | {NUM_CLASSES} clases | '
             f'{len(all_samples):,} imgs | Mejor F1: {best_f1:.4f}', fontsize=12)

axes[0].plot(ep_r, history['train_loss'], 'b-o', ms=3, label='Train')
axes[0].plot(ep_r, history['val_loss'],   'r-o', ms=3, label='Val')
axes[0].axvline(NUM_EPOCHS_P1 + 0.5, color='gray', ls='--', lw=1.5, label='→ Fase 2')
axes[0].set_title('Loss'); axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].plot(ep_r, history['val_f1'], 'g-o', ms=3)
axes[1].axvline(NUM_EPOCHS_P1 + 0.5, color='gray', ls='--', lw=1.5)
axes[1].axhline(0.7, color='green', ls='--', lw=1.5, label='Meta buena')
axes[1].set_title('Macro F1'); axes[1].set_ylim(0, 1)
axes[1].legend(); axes[1].grid(alpha=0.3)

for ax in axes:
    ax.set_xlabel('Época')
plt.tight_layout()
plt.savefig(SAVE_PATH.replace('.pt', '_grafica.png'), dpi=100, bbox_inches='tight')
plt.show()

# ─── F1 por clase (validación con imágenes reales) ────────────
_, final_f1, f1_per_class = eval_epoch(model, val_loader)
print("\n" + "═" * 60)
print("📊 F1 POR CLASE (validation)")
print("═" * 60)
for cls, f1 in zip(CLASSES, f1_per_class):
    alerta = " ⚠️" if f1 < 0.5 else ""
    print(f"  {cls:22s}: {f1:.4f}  {'█'*int(f1*20)}{alerta}")
print(f"\n  {'MACRO F1':22s}: {final_f1:.4f}")
print(f"\nModelo: {SAVE_PATH}")